# LangChain + Sarvam RAG

A Retrieval-Augmented Generation pipeline that uses LangChain for vector search
and Sarvam AI `sarvam-m` for answer generation.

## Pipeline

```
User query
        |
        v
retrieve_documents()    <- LangChain FAISS + HuggingFaceEmbeddings
        |  query embedded with paraphrase-multilingual-mpnet-base-v2
        |  vector_store.as_retriever().invoke(query)
        v
generate_answer()       <- Sarvam sarvam-m (chat.completions)
        |  context: page_content from retrieved Documents
        v
Answer
```

In [ ]:
%pip install sarvamai>=0.1.26 langchain>=0.3.0 langchain-community>=0.3.0 sentence-transformers>=2.2.2 faiss-cpu>=1.7.4 python-dotenv>=1.0.0

## Setup and API Key

Load environment variables and initialise the Sarvam AI client.
Set `SARVAM_API_KEY` in a `.env` file copied from `.env.example`.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
from sarvamai import SarvamAI

load_dotenv()

SARVAM_API_KEY = os.environ.get("SARVAM_API_KEY", "")
if not SARVAM_API_KEY:
    raise RuntimeError(
        "SARVAM_API_KEY is not set. "
        "Copy .env.example to .env and paste your key."
    )

client = SarvamAI(api_subscription_key=SARVAM_API_KEY)
EMBEDDING_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
print("Setup complete.")

## Sample Documents

Create two in-memory LangChain `Document` objects.
Replace these with your own data loaded from files, databases, or APIs.

In [ ]:
SAMPLE_DOCS: list[Document] = [
    Document(
        page_content=(
            "Sarvam AI is an Indian AI company building large language models and APIs "
            "optimised for Indian languages. Its offerings include speech-to-text, "
            "text-to-speech, translation, and chat completion APIs that support languages "
            "such as Hindi, Tamil, Telugu, Kannada, Bengali, and more."
        ),
        metadata={"source": "doc-1", "topic": "sarvam-ai"},
    ),
    Document(
        page_content=(
            "LangChain is an open-source framework for building applications powered by "
            "large language models. It provides abstractions for document loading, text "
            "splitting, embedding, vector storage, and retrieval chains. The FAISS "
            "integration allows fast in-memory similarity search over embedded documents."
        ),
        metadata={"source": "doc-2", "topic": "langchain"},
    ),
]

print(f"Created {len(SAMPLE_DOCS)} sample documents.")
for doc in SAMPLE_DOCS:
    preview = doc.page_content[:80].replace("\n", " ")
    print(f"  [{doc.metadata['source']}] {preview}...")

## Build FAISS Vector Store

Index the sample documents using LangChain `FAISS.from_documents`.
The `HuggingFaceEmbeddings` wrapper calls the sentence-transformers model locally.

Model: `paraphrase-multilingual-mpnet-base-v2` (~420 MB first download, cached after that)

In [ ]:
def create_vector_store(
    documents: list[Document],
    model_name: str = EMBEDDING_MODEL,
) -> FAISS:
    """Build a LangChain FAISS vector store from a list of Documents.

    Args:
        documents: List of LangChain Document objects to index.
        model_name: HuggingFace model name used for sentence-transformers embeddings.

    Returns:
        Populated FAISS vector store ready for similarity search.
    """
    embeddings = HuggingFaceEmbeddings(model_name=model_name)
    return FAISS.from_documents(documents, embeddings)

In [ ]:
vector_store = create_vector_store(SAMPLE_DOCS)

print(f"Vector store built.")
print(f"Embedding model  : {EMBEDDING_MODEL}")
print(f"Documents indexed: {vector_store.index.ntotal}")